In [ ]:
from google.colab import files
files.upload()

Saving app.py to app.py
Saving requirements.txt to requirements.txt


{'app.py': b'import io\r\nimport os\r\nimport re\r\nimport tempfile\r\nimport time\r\n\r\nimport streamlit as st\r\nimport torch\r\nfrom diffusers import DDIMScheduler, StableDiffusionPipeline\r\nfrom faster_whisper import WhisperModel\r\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer\r\n\r\n# =========================================================\r\n# Page config + styling\r\n# =========================================================\r\nst.set_page_config(page_title="Voice \xe2\x86\x92 Image Studio", page_icon="\xf0\x9f\x8e\xa8", layout="wide")\r\n\r\nst.markdown(\r\n    """\r\n    <style>\r\n    .hero {\r\n        padding: 1.6rem 2rem; border-radius: 18px; margin-bottom: 1.2rem;\r\n        background: linear-gradient(120deg, #4f46e5 0%, #9333ea 55%, #ec4899 100%);\r\n        color: white;\r\n    }\r\n    .hero h1 { margin: 0; font-size: 2rem; color: white; }\r\n    .hero p  { margin: .3rem 0 0 0; opacity: .92; }\r\n    .step-title { font-weight: 700; font-size: 1.0

In [ ]:
!pip install -q streamlit diffusers transformers accelerate sentencepiece faster-whisper librosa soundfile
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

True Tesla T4


In [ ]:
import subprocess, time, re, os, glob

# Find the folders that contain the CUDA 12 libs
lib_dirs = set()
for pattern in ("libcublas.so.12", "libcudnn.so.9"):
    for p in glob.glob(f"/usr/local/lib/python3*/dist-packages/nvidia/**/{pattern}", recursive=True):
        lib_dirs.add(os.path.dirname(p))
print("Library folders:", lib_dirs)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(lib_dirs) + ":" + env.get("LD_LIBRARY_PATH", "")

subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.headless", "true", "--server.enableCORS", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=env,
)
time.sleep(8)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Open this link:", m.group(0))
        break

Library folders: {'/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib'}
Open this link: https://jobs-weddings-markers-launched.trycloudflare.com


In [ ]:
!pkill -f streamlit
!pkill -f cloudflared